# Keyword-Based Disease Annotator

Annotate `data/processed/merged_data.csv` using keyword presence in `cleaned_text`.

**Label order contract:** `["AURI", "PN", "TB", "COVID"]`

This notebook:
- loads base keyword CSVs from `docs/keywords/`
- normalizes keywords and `cleaned_text`
- creates `cleaned_text`, `disease`, `misinformation`, and `sentiment` columns for merged data
- keeps no-disease rows as `[0,0,0,0]`
- writes `data/training_data/merged_data_annotated.csv`
- also writes `data/training_data/training_1_annotated.csv` with `training_1.csv` disease annotations preserved and only `misinformation`/`sentiment` added or replaced


In [1]:
import csv
import json
from pathlib import Path

import pandas as pd

LABELS = ["AURI", "PN", "TB", "COVID"]
OUTPUT_COLUMNS = ["cleaned_text", "disease", "misinformation", "sentiment"]
TRAINING_1_AUX_COLUMNS = ["misinformation", "sentiment"]
CWD = Path.cwd()
ROOT_DIR = next((path for path in [CWD, *CWD.parents] if (path / ".git").exists()), CWD)
KEYWORD_FILES = {
    "AURI": ROOT_DIR / "docs/keywords/ri_keywords.csv",
    "PN": ROOT_DIR / "docs/keywords/pn_keywords.csv",
    "TB": ROOT_DIR / "docs/keywords/tb_keywords.csv",
    "COVID": ROOT_DIR / "docs/keywords/covid_keywords.csv",
}
INPUT_PATH = ROOT_DIR / "data/processed/merged_data.csv"
OUTPUT_PATH = ROOT_DIR / "data/training_data/training_2.csv"
TRAINING_1_INPUT_PATH = ROOT_DIR / "data/training_data/gold_standard.csv"
TRAINING_1_OUTPUT_PATH = ROOT_DIR / "data/training_data/training_1.csv"
TRAINING_1_TEXT_COLUMN = "post"
TRAINING_1_DISEASE_COLUMN = "annotate"


In [2]:
def normalize_text(value):
    if value is None:
        return ""
    text = str(value).strip().lower()
    return "" if text == "nan" else text


def load_csv_rows(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset: {path}")

    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        return list(reader), reader.fieldnames or []


def load_keywords(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing keyword file: {path}")

    keywords = []
    seen = set()

    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        for row in reader:
            for cell in row:
                keyword = normalize_text(cell)
                if keyword and keyword not in seen:
                    keywords.append(keyword)
                    seen.add(keyword)

    if not keywords:
        raise ValueError(f"Keyword file is empty after normalization: {path}")

    return keywords


def validate_disease_vector(value, source):
    try:
        vector = json.loads(value)
    except (TypeError, json.JSONDecodeError) as exc:
        raise ValueError(f"Invalid disease vector in {source}: {value!r}") from exc

    if not (
        isinstance(vector, list)
        and len(vector) == len(LABELS)
        and all(item in (0, 1) for item in vector)
    ):
        raise ValueError(f"Invalid disease vector in {source}: {value!r}")

    return vector


def annotate_text(cleaned_text, keyword_map):
    text = normalize_text(cleaned_text)
    vector = [int(any(keyword in text for keyword in keyword_map[label])) for label in LABELS]
    return vector


In [3]:
rows, fieldnames = load_csv_rows(INPUT_PATH)
training_1_rows, training_1_fieldnames = load_csv_rows(TRAINING_1_INPUT_PATH)

if "cleaned_text" not in fieldnames:
    raise KeyError("Expected `cleaned_text` column in the source dataset.")

for required_column in [TRAINING_1_TEXT_COLUMN, TRAINING_1_DISEASE_COLUMN]:
    if required_column not in training_1_fieldnames:
        raise KeyError(f"Expected `{required_column}` column in training_1.csv.")

keyword_map = {label: load_keywords(path) for label, path in KEYWORD_FILES.items()}
keyword_counts = {label: len(keywords) for label, keywords in keyword_map.items()}

print({
    "cwd": str(CWD),
    "root_dir": str(ROOT_DIR),
    "merged_rows": len(rows),
    "merged_input_path": str(INPUT_PATH),
    "merged_output_path": str(OUTPUT_PATH),
    "training_1_rows": len(training_1_rows),
    "training_1_input_path": str(TRAINING_1_INPUT_PATH),
    "training_1_output_path": str(TRAINING_1_OUTPUT_PATH),
    "keyword_counts": keyword_counts,
})


{'cwd': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/notebooks', 'root_dir': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus', 'merged_rows': 39458, 'merged_input_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/processed/merged_data.csv', 'merged_output_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/training_data/training_2.csv', 'training_1_rows': 21968, 'training_1_input_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/training_data/gold_standard.csv', 'training_1_output_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/training_data/training_1.csv', 'keyword_counts': {'AURI': 71, 'PN': 80, 'TB': 78, 'COVID': 115}}


In [4]:
annotated_rows = []
for row in rows:
    vector = annotate_text(row.get("cleaned_text", ""), keyword_map)
    annotated_rows.append({
        "cleaned_text": row.get("cleaned_text", ""),
        "disease": json.dumps(vector, separators=(",", ":")),
        "misinformation": 0,
        "sentiment": 0,
    })

annotated_df = pd.DataFrame(annotated_rows, columns=OUTPUT_COLUMNS)

print({"output_rows": len(annotated_df), "output_columns": list(annotated_df.columns)})
display(annotated_df.head())


{'output_rows': 39458, 'output_columns': ['cleaned_text', 'disease', 'misinformation', 'sentiment']}


,cleaned_text,disease,misinformation,sentiment
0,p 11,"[0,0,0,0]",0,0
1,anong gusto mo gawin niya makipagbarda sa mga ...,"[1,1,1,1]",0,0
2,grabe na hutoy sa ubo thanks mama cels sa pag ...,"[1,1,1,1]",0,0
3,trautman sucks ass,"[0,0,0,0]",0,0
4,thoughts on the officiating crew tonight? that...,"[0,0,0,0]",0,0


In [5]:
parsed_vectors = annotated_df["disease"].apply(json.loads).tolist()
vector_lengths = [len(values) for values in parsed_vectors]
binary_ok = [all(item in (0, 1) for item in values) for values in parsed_vectors]

assert list(annotated_df.columns) == OUTPUT_COLUMNS
assert len(annotated_df) == len(rows)
assert all(length == len(LABELS) for length in vector_lengths)
assert all(binary_ok)
assert annotated_df["misinformation"].isin([0, 1]).all()
assert annotated_df["sentiment"].isin([0, 1]).all()

print("Annotation contract validated.")
print({"input_rows": len(rows), "output_rows": len(annotated_df)})
print(pd.DataFrame(parsed_vectors, columns=LABELS).sum().astype(int).to_dict())


Annotation contract validated.
{'input_rows': 39458, 'output_rows': 39458}
{'AURI': 24819, 'PN': 9726, 'TB': 10498, 'COVID': 13613}


In [6]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
annotated_df.to_csv(OUTPUT_PATH, index=False)
print(f"Exported {len(annotated_df):,} annotated rows to {OUTPUT_PATH}")


Exported 39,458 annotated rows to /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/training_data/training_2.csv


In [7]:
# Preserve curated disease annotations from training_1.csv exactly; only auxiliary labels change.
training_1_annotated_df = pd.DataFrame(training_1_rows)
training_1_disease_before = training_1_annotated_df[TRAINING_1_DISEASE_COLUMN].copy()

for row_index, value in training_1_disease_before.items():
    validate_disease_vector(value, source=f"training_1.csv row {row_index + 2}")

for column in TRAINING_1_AUX_COLUMNS:
    training_1_annotated_df[column] = 0

assert training_1_annotated_df[TRAINING_1_DISEASE_COLUMN].tolist() == training_1_disease_before.tolist()
assert training_1_annotated_df["misinformation"].isin([0, 1]).all()
assert training_1_annotated_df["sentiment"].isin([0, 1]).all()

print({
    "training_1_output_rows": len(training_1_annotated_df),
    "training_1_output_columns": list(training_1_annotated_df.columns),
    "disease_column_preserved": TRAINING_1_DISEASE_COLUMN,
})
display(training_1_annotated_df.head())


{'training_1_output_rows': 21968, 'training_1_output_columns': ['post', 'annotate', 'misinformation', 'sentiment'], 'disease_column_preserved': 'annotate'}


,post,annotate,misinformation,sentiment
0,Idk why everyone is getting cough colds but pl...,"[1, 0, 0, 1]",0,0
1,nanghihina na yung tao oh,"[0, 0, 0, 0]",0,0
2,fever dream,"[0, 0, 0, 0]",0,0
3,nagsusulat ako sa kalagitnaan ng pagtanto na a...,"[0, 0, 0, 0]",0,0
4,tgiff thank goodness its fever friday! join us...,"[0, 0, 0, 0]",0,0


In [8]:
TRAINING_1_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
training_1_annotated_df.to_csv(TRAINING_1_OUTPUT_PATH, index=False)
print(f"Exported {len(training_1_annotated_df):,} training_1 rows to {TRAINING_1_OUTPUT_PATH}")


Exported 21,968 training_1 rows to /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/training_data/training_1.csv


In [9]:
# Spot checks for single-label rows by disease.
eda_columns = ["cleaned_text", "disease", "misinformation", "sentiment"]
disease_vectors = pd.DataFrame(annotated_df["disease"].apply(json.loads).tolist(), columns=LABELS)
single_label_mask = disease_vectors.sum(axis=1).eq(1)

def sample_label_only(label, n=3):
    label_mask = disease_vectors[label].eq(1) & single_label_mask
    label_rows = annotated_df.loc[label_mask, eda_columns]
    return label_rows.sample(n=min(n, len(label_rows)), random_state=42)

label_sample_dfs = {label: sample_label_only(label) for label in LABELS}

for label, label_sample_df in label_sample_dfs.items():
    print(f"{label}-only examples")
    display(label_sample_df)


AURI-only examples


,cleaned_text,disease,misinformation,sentiment
28465,i like her sm but i have priorities i [23m] me...,"[1,0,0,0]",0,0
23336,psu (power supply) aging? haluu! i'm not super...,"[1,0,0,0]",0,0
24415,"27 [m4f] tatagos ka ba ya hi! so recently, pur...","[1,0,0,0]",0,0


PN-only examples


,cleaned_text,disease,misinformation,sentiment
3568,these are some possible causes: * sweating and...,"[0,1,0,0]",0,0
3875,sofer naman ang katugnaw,"[0,1,0,0]",0,0
6239,lahug to mandaue to lapu-lapu tas adto napod s...,"[0,1,0,0]",0,0


TB-only examples


,cleaned_text,disease,misinformation,sentiment
27780,vincents eatery tambo nakatry na mig kaon sa v...,"[0,0,1,0]",0,0
4495,may ex bf si bella kaka break lang 2025 nung n...,"[0,0,1,0]",0,0
7845,wts batu evolved 1.5k/ea take 20 free 2 sc tb ...,"[0,0,1,0]",0,0


COVID-only examples


,cleaned_text,disease,misinformation,sentiment
14917,yhanieyeah selfcare 08/09/25 girl if you wanna...,"[0,0,0,1]",0,0
32608,help paano ba maiwasan ang pagsusuka sa byahe?,"[0,0,0,1]",0,0
16219,"ordered some covid tests from amazon, got a fr...","[0,0,0,1]",0,0
